In [1]:
pip install -r requirments.txt


[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
!aws configure 
# This can be done via terminal there we have input box to enter access key and all keys.

AWS Access Key ID [****************YWM3]: ^C


In [27]:
params={
    'model_name': 'distilbert-base-cased',
    'learning_rate':5e-5,
    'batch_size':16,
    'num_epoch':1,
    'dataset_name':'ag_news',
    'task_name':'sequence_classification',
    'log_steps':100,
    'max_seq_length':128,
    'output_dir':'model/distilbert-base-uncased-ag_news'
    }

In [40]:
import mlflow

mlflow.set_tracking_uri("http://ec2-34-203-192-182.compute-1.amazonaws.com:5000/")
mlflow.set_experiment(f"{params['task_name']}")

2026/02/14 23:03:31 INFO mlflow.tracking.fluent: Experiment with name 'sequence_classification' does not exist. Creating a new experiment.


<Experiment: artifact_location='s3://ml-flow-2-9802/4', creation_time=1771090411379, experiment_id='4', last_update_time=1771090411379, lifecycle_stage='active', name='sequence_classification', tags={}>

In [37]:
import os
import mlflow
from sklearn.metrics import accuracy_score,precision_recall_fscore_support
import torch
from tqdm import tqdm
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import DistilBertForSequenceClassification, DistilBertTokenizer

In [11]:
dataset=load_dataset(params['dataset_name'])
dataset


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

In [24]:
dataset['train'][0:5]

{'text': ["Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
  'Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group,\\which has a reputation for making well-timed and occasionally\\controversial plays in the defense industry, has quietly placed\\its bets on another part of the market.',
  "Oil and Economy Cloud Stocks' Outlook (Reuters) Reuters - Soaring crude prices plus worries\\about the economy and the outlook for earnings are expected to\\hang over the stock market next week during the depth of the\\summer doldrums.",
  'Iraq Halts Oil Exports from Main Southern Pipeline (Reuters) Reuters - Authorities have halted oil export\\flows from the main pipeline in southern Iraq after\\intelligence showed a rebel militia could strike\\infrastructure, an oil official said on Saturday.',
  'Oil prices soar to all-time record, posing new menace to 

In [31]:
# Load and preprocess dataset
dataset = load_dataset(params['dataset_name'])
tokenizer = DistilBertTokenizer.from_pretrained(params['model_name'])

def tokenize(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=params['max_seq_length'])

train_dataset = dataset["train"].shuffle().select(range(20_000)).map(tokenize, batched=True)
test_dataset = dataset["test"].shuffle().select(range(2_000)).map(tokenize, batched=True)
train_dataset
# Set format for PyTorch and create data loaders
train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format('torch', columns=['input_ids', 'attention_mask' , 'label'])

train_loader = DataLoader(train_dataset, batch_size=params['batch_size'], shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=params['batch_size'], shuffle=False)
# get the labels
labels = dataset["train"].features['label'].names
labels

Map: 100%|██████████| 2000/2000 [00:00<00:00, 32311.60 examples/s]


['World', 'Sports', 'Business', 'Sci/Tech']

In [32]:
model = DistilBertForSequenceClassification.from_pretrained(params['model_name'], num_labels=len (labels))
model.config.id2label = {i: label for i, label in enumerate(labels)}
params['id2label'] = model.config.id2label
device = torch.device("cuda" if torch. cuda.is_available() else "cpu")
model.to(device)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2218.68it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [36]:
from torch.optim import AdamW

optimizer = AdamW(model. parameters(), lr=params['learning_rate'])

In [43]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_model(model, dataloader, device):
    model.eval()    # Set model to evaluation mode
    predictions, true_labels = [],[]

    with torch.no_grad():
        for batch in dataloader:
            inputs, masks, labels = batch['input_ids'].to(device), batch['attention_mask'].to(device), batch['label'].to(device)
            
            # Forward pass, calculate logit predictions
            outputs = model(inputs, attention_mask=masks)
            logits = outputs.logits
            _, predicted_labels = torch.max(logits, dim=1)

            predictions.extend(predicted_labels.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(true_labels, predictions)
    precision = precision_score(true_labels, predictions, average='weighted')
    recall = recall_score(true_labels, predictions, average='weighted')
    f1 = f1_score(true_labels, predictions, average='weighted')

    return accuracy, precision, recall, f1

    # # Calculate Evaluation Metrics
    # accuracy = accuracy_score(true_labels, predictions)

In [46]:
import os
import mlflow
import torch
from tqdm import tqdm

# Start MLflow Run
with mlflow.start_run(
    run_name=f"{params['model_name']}-{params['dataset_name']}"
) as run:

    # Log parameters
    mlflow.log_params(params)

    model.train()

    total_steps = params['num_epoch'] * len(train_loader)

    with tqdm(total=total_steps, desc="Training") as pbar:

        for epoch in range(params['num_epoch']):

            running_loss = 0.0

            for i, batch in enumerate(train_loader):

                inputs = batch['input_ids'].to(device)
                masks = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)

                optimizer.zero_grad()

                outputs = model(
                    inputs,
                    attention_mask=masks,
                    labels=labels
                )

                loss = outputs.loss
                loss.backward()
                optimizer.step()

                running_loss += loss.item()

                # Log every N steps
                if (i + 1) % params['log_steps'] == 0:
                    avg_loss = running_loss / params['log_steps']

                    mlflow.log_metric(
                        "loss",
                        avg_loss,
                        step=epoch * len(train_loader) + i
                    )

                    running_loss = 0.0

                pbar.update(1)

            # 🔎 Evaluate after each epoch
            accuracy, precision, recall, f1 = evaluate_model(
                model, test_loader, device
            )

            print(
                f"Epoch {epoch + 1} | "
                f"Accuracy: {accuracy:.4f} | "
                f"Precision: {precision:.4f} | "
                f"Recall: {recall:.4f} | "
                f"F1: {f1:.4f}"
            )

            # Log metrics
            mlflow.log_metrics({
                "accuracy": accuracy,
                "precision": precision,
                "recall": recall,
                "f1": f1
            }, step=epoch)


    # Save locally (optional)
    os.makedirs(params['output_dir'], exist_ok=True)
    model.save_pretrained(params['output_dir'])
    tokenizer.save_pretrained(params['output_dir'])

    # Log as MLflow Model (IMPORTANT)
    mlflow.pytorch.log_model(
        pytorch_model=model,
        artifact_path="model"
    )

    # Register Model
    model_uri = f"runs:/{run.info.run_id}/model"
    mlflow.register_model(model_uri, "agnews-transformer")

print("Finished Training")


Training: 100%|██████████| 1250/1250 [16:17<00:00,  1.25it/s]

Epoch 1 | Accuracy: 0.9185 | Precision: 0.9193 | Recall: 0.9185 | F1: 0.9184


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 10.63it/s]
2026/02/15 00:06:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/15 00:06:09 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.
Registered model 'agnews-transformer' already exists. Creating a new version of this model...
2026/02/15 00:06:54 WARNING mlflow.tracking._model_registry.fluent: Run with id 47730398a8484037a2f4ff30a5cf0be2 has no artifacts at artifact path 'model', registering model based on models:/m-12b8f3d848ab45698f2ddf82b0573bfa instead
2026/02/15 00:06:55 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to fi

🏃 View run distilbert-base-cased-ag_news at: http://ec2-34-203-192-182.compute-1.amazonaws.com:5000/#/experiments/4/runs/47730398a8484037a2f4ff30a5cf0be2
🧪 View experiment at: http://ec2-34-203-192-182.compute-1.amazonaws.com:5000/#/experiments/4
Finished Training
